# Assignment 3 — Part C
## Hasan Syed - 100625483
## 10/23/2026

## Tracking a Deep Learning Experiment with MLflow and DagsHub (MNIST)

**Before running:**
1. Create a free account at https://dagshub.com
2. Create a repository (e.g. `mnist-mlflow-tracking`), or connect your GitHub repo to DagsHub
3. Set `DAGSHUB_USER` / `DAGSHUB_REPO` in the setup cell
4. When `dagshub.init()` runs it shows a link — click it to authorize.

!pip install -q dagshub mlflow

In [4]:
import dagshub
import mlflow
import mlflow.tensorflow
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

### 1. DagsHub + MLflow setup

In [5]:
DAGSHUB_USER = "hasan.syed1"   # <-- change me
DAGSHUB_REPO = "mnist-mlflow-tracking"   # <-- change me

dagshub.init(repo_owner=DAGSHUB_USER, repo_name=DAGSHUB_REPO, mlflow=True)

EXPERIMENT_NAME = "MNIST-Dense-NN"
mlflow.set_experiment(EXPERIMENT_NAME)
print("MLflow Tracking URI: https://dagshub.com/hasan.syed1/mnist-mlflow-tracking", mlflow.get_tracking_uri())

Initialized MLflow to track repo "hasan.syed1/mnist-mlflow-tracking"

Repository hasan.syed1/mnist-mlflow-tracking initialized!

MLflow Tracking URI: https://dagshub.com/hasan.syed1/mnist-mlflow-tracking https://dagshub.com/hasan.syed1/mnist-mlflow-tracking.mlflow


### 2. Enable TensorFlow/Keras autologging

In [6]:
mlflow.tensorflow.autolog(log_models=True)

### 3. Load and preprocess MNIST

In [7]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.astype("float32") / 255.0   # scale pixels to [0, 1]
x_test = x_test.astype("float32") / 255.0
print("Train:", x_train.shape, "Test:", x_test.shape)

VALIDATION_SPLIT = 0.1  # last 10% of training data (6,000 images) used for validation

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Train: (60000, 28, 28) Test: (10000, 28, 28)


### 4. Model builder (baseline = Flatten -> Dense(128, relu) -> Dense(10, softmax))

In [8]:
def build_model(hidden_layers, optimizer_name):
    model = keras.Sequential([keras.Input(shape=(28, 28)), layers.Flatten()])
    for units in hidden_layers:
        model.add(layers.Dense(units, activation="relu"))
    model.add(layers.Dense(10, activation="softmax"))
    model.compile(
        optimizer=optimizer_name,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

### 5-9. Experiments: baseline + additional runs, each with a meaningful name

In [9]:
experiments = [
    # Baseline required by the assignment
    {"run_name": "baseline_1x128_adam_bs64_ep50",
     "hidden_layers": [128], "optimizer": "adam", "batch_size": 64, "epochs": 50},
    # Deeper network
    {"run_name": "deeper_2x(256-128)_adam_bs64_ep50",
     "hidden_layers": [256, 128], "optimizer": "adam", "batch_size": 64, "epochs": 50},
    # Different optimizer
    {"run_name": "sgd_1x128_bs64_ep50",
     "hidden_layers": [128], "optimizer": "sgd", "batch_size": 64, "epochs": 50},
    # Wider layer, larger batch, fewer epochs
    {"run_name": "wide_1x512_adam_bs256_ep20",
     "hidden_layers": [512], "optimizer": "adam", "batch_size": 256, "epochs": 20},
    # RMSprop, smaller batch, fewer epochs
    {"run_name": "rmsprop_1x128_bs32_ep15",
     "hidden_layers": [128], "optimizer": "rmsprop", "batch_size": 32, "epochs": 15},
]

for cfg in experiments:
    tf.keras.backend.clear_session()
    tf.random.set_seed(42)

    with mlflow.start_run(run_name=cfg["run_name"]):
        # Required manual parameters (step 6)
        mlflow.log_params({
            "num_hidden_layers": len(cfg["hidden_layers"]),
            "nodes_per_hidden_layer": str(cfg["hidden_layers"]),
            "optimizer": cfg["optimizer"],
            "batch_size": cfg["batch_size"],
            "epochs": cfg["epochs"],
        })

        model = build_model(cfg["hidden_layers"], cfg["optimizer"])
        history = model.fit(
            x_train, y_train,
            batch_size=cfg["batch_size"],
            epochs=cfg["epochs"],
            validation_split=VALIDATION_SPLIT,
            verbose=2,
        )

        # Summary metrics that make the run comparison table easy to read
        h = history.history
        best_epoch = int(max(range(len(h["val_accuracy"])), key=lambda i: h["val_accuracy"][i]))
        test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
        mlflow.log_metrics({
            "final_train_accuracy": h["accuracy"][-1],
            "final_val_accuracy": h["val_accuracy"][-1],
            "final_train_loss": h["loss"][-1],
            "final_val_loss": h["val_loss"][-1],
            "best_val_accuracy": h["val_accuracy"][best_epoch],
            "best_val_epoch": best_epoch + 1,
            "min_val_loss": min(h["val_loss"]),
            "test_accuracy": test_acc,
            "test_loss": test_loss,
        })
        print(f"{cfg['run_name']}: val_acc={h['val_accuracy'][-1]:.4f} test_acc={test_acc:.4f}")

Epoch 1/50


844/844 - 11s - 13ms/step - accuracy: 0.9130 - loss: 0.3088 - val_accuracy: 0.9587 - val_loss: 0.1410
Epoch 2/50


844/844 - 5s - 6ms/step - accuracy: 0.9587 - loss: 0.1424 - val_accuracy: 0.9692 - val_loss: 0.1023
Epoch 3/50


844/844 - 6s - 7ms/step - accuracy: 0.9711 - loss: 0.1002 - val_accuracy: 0.9728 - val_loss: 0.0865
Epoch 4/50


844/844 - 6s - 7ms/step - accuracy: 0.9777 - loss: 0.0755 - val_accuracy: 0.9747 - val_loss: 0.0803
Epoch 5/50


...

1688/1688 - 9s - 6ms/step - accuracy: 0.9746 - loss: 0.0879 - val_accuracy: 0.9755 - val_loss: 0.0887
Epoch 4/15


1688/1688 - 10s - 6ms/step - accuracy: 0.9803 - loss: 0.0695 - val_accuracy: 0.9773 - val_loss: 0.0858
Epoch 5/15
1688/1688 - 8s - 5ms/step - accuracy: 0.9840 - loss: 0.0570 - val_accuracy: 0.9782 - val_loss: 0.0860
Epoch 6/15
1688/1688 - 11s - 7ms/step - accuracy: 0.9872 - loss: 0.0478 - val_accuracy: 0.9787 - val_loss: 0.0866
Epoch 7/15
1688/1688 - 7s - 4ms/step - accuracy: 0.9894 - loss: 0.0410 - val_accuracy: 0.9785 - val_loss: 0.0899
Epoch 8/15
1688/1688 - 14s - 8ms/step - accuracy: 0.9912 - loss: 0.0354 - val_accuracy: 0.9778 - val_loss: 0.0955
Epoch 9/15
1688/1688 - 9s - 5ms/step - accuracy: 0.9926 - loss: 0.0306 - val_accuracy: 0.9780 - val_loss: 0.0977
Epoch 10/15
1688/1688 - 8s - 5ms/step - accuracy: 0.9938 - loss: 0.0266 - val_accuracy: 0.9782 - val_loss: 0.1009
Epoch 11/15
1688/1688 - 9s - 5ms/step - accuracy: 0.9946 - loss: 0.0234 - val_accuracy: 0.9778 - val_loss: 0.1061
Epoch 12/15
1688/1688 - 15s - 9ms/step - accuracy: 0.9953 - loss: 0.0203 - val_accuracy: 0.9775 - val_

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 194ms/step


2026/10/23 22:36:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


rmsprop_1x128_bs32_ep15: val_acc=0.9783 test_acc=0.9752
🏃 View run rmsprop_1x128_bs32_ep15 at: https://dagshub.com/hasan.syed1/mnist-mlflow-tracking.mlflow/#/experiments/0/runs/536af1012e1d47eeaa1fffa5be6ad6e6
🧪 View experiment at: https://dagshub.com/hasan.syed1/mnist-mlflow-tracking.mlflow/#/experiments/0


### 10-11. Compare runs and pick the best one by validation performance

In [10]:
runs = mlflow.search_runs(experiment_names=[EXPERIMENT_NAME])
cols = [
    "tags.mlflow.runName",
    "params.num_hidden_layers", "params.nodes_per_hidden_layer",
    "params.optimizer", "params.batch_size", "params.epochs",
    "metrics.final_train_accuracy", "metrics.final_val_accuracy",
    "metrics.final_train_loss", "metrics.final_val_loss",
    "metrics.best_val_accuracy", "metrics.min_val_loss", "metrics.test_accuracy",
]
summary = runs[[c for c in cols if c in runs.columns]].copy()
summary.columns = [c.split(".", 1)[1] for c in summary.columns]
summary = summary.sort_values("final_val_accuracy", ascending=False)

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)
print("\n=== Run comparison (sorted by final validation accuracy) ===")
print(summary.to_string(index=False))
summary.to_csv("run_comparison.csv", index=False)

summary = summary.rename(columns={"mlflow.runName": "run_name"})
best = summary.iloc[0]
print(f"\nBest run: {best['run_name']}")
print(f"  final val accuracy = {best['final_val_accuracy']:.4f}, "
      f"final val loss = {best['final_val_loss']:.4f}")
print("\nOpen your experiments at:")
print(f"  https://dagshub.com/hasan.syed1/mnist-mlflow-tracking.mlflow")


=== Run comparison (sorted by final validation accuracy) ===
                   mlflow.runName num_hidden_layers nodes_per_hidden_layer optimizer batch_size epochs  final_train_accuracy  final_val_accuracy  final_train_loss  final_val_loss  best_val_accuracy  min_val_loss  test_accuracy
       wide_1x512_adam_bs256_ep20                 1                  [512]      adam        256     20              0.999963            0.985500          0.002113        0.071233           0.986167      0.065964         0.9821
    baseline_1x128_adam_bs64_ep50                 1                  [128]      adam         64     50              0.996685            0.981000          0.011173        0.131010           0.983833      0.073279         0.9778
deeper_2x(256-128)_adam_bs64_ep50                 2             [256, 128]      adam         64     50              0.998444            0.981000          0.004959        0.188496           0.985333      0.085085         0.9775
          rmsprop_1x128_bs32_e